In [1]:
import pandas as pd
import os
from pathlib import Path

In [2]:
current_dir = Path(os.getcwd())
if current_dir.name == "preprocessing":
    project_root = current_dir.parent
else:
    project_root = current_dir
    
input_path = project_root / 'data' / 'stanford-hCMST.tsv'
output_path = project_root / 'data' / 'hCMST_filtered.csv'

df = pd.read_csv(input_path, sep='\t')

In [3]:
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (3510, 725)


,CASEID_NEW,W3_WEIGHT,W3_WEIGHT_LGB,W3_COMBO_WEIGHT,W3_ATTRITION_ADJ_WEIGHT,W2_WEIGHT_GENPOP,W2_WEIGHT_LGB,W2_COMBO_WEIGHT,W2_ATTRITION_ADJ_WEIGHTS,W1_WEIGHT_COMBO,...,P20_PPPA1634,P20_PPPA1902,P20_PPPA1903,P20_PPPA1904,P20_PPP22001,P20_PPPA1905,P20_PPPA1648,P20_PPP20072,P20_PPP20071,P20_PPP2DATE2020
0,53001,.4422,,.4953084588051,.400184631348,.3856,,.4376695454121,.3803511857986,0.426861,...,2,0,1,0,0,0,13,6,,20210506
1,71609,.8284,,.9278913140297,.879257798195,.9196,,1.0437783002853,.9539480209351,1.295508,...,2,0,1,0,0,0,2,5,1,20201118
2,106983,.8255,,.9246429800987,.7064666152,.7748,,.8794252276421,.7246816754341,1.126573,...,1,1,0,0,0,0,1,4,2,20210429
3,121759,,,,,.9177,,1.0416216850281,.7930925488472,0.933440,...,1,1,0,0,0,0,11,2,1,20210507
4,158083,.881,,.9868085384369,.655467092991,.8697,,.9871400594711,.7354734539986,0.931291,...,1,1,0,0,0,0,13,6,,20210602


In [ ]:
target_columns_dict = {
    'W1_AGE_WHEN_MET': "Subject age",
    'W1_Q9': "Partner age", 
    'W1_SUBJECT_RACE': "Subject race",
    'W1_Q6B': "Partner race",
    'W1_INTERRACIAL_5CAT': "Interracial relationship",
    'W1_Q6A': "Hispanic origin of partner",
    'W1_PPETHM': "Subject Hispanic origin", 
    'W1_PPGENDER': "Subject gender",
    'W1_Q4': "Partner gender",
    'W1_ATTRACTION': "Attraction",
    'W1_PPEDUC': "Subject education",
    'W1_Q10': "Partner education",
    'W1_PPREG9': "Region",
    'W1_PPWORK': "Subject's employment status", 
    'W1_Q23': "Earn more money?",
    'W1_Q25': "Same high school?",
    'W1_Q26': "Same college?",
    'W1_Q27': "Grew up same city?", 
    'W1_Q19': "Living Together", 
    'W1_MARRIED': "Married at baseline",
    'W1_Q34': "Quality of Relationship?" 
}

Filter Down to only participant Info and Answers

In [166]:
filtered_columns = list(target_columns_dict.keys()) 
df_w1_only = df[filtered_columns]

In [167]:
df_w1_only

,W1_AGE_WHEN_MET,W1_Q9,W1_SUBJECT_RACE,W1_Q6B,W1_INTERRACIAL_5CAT,W1_Q6A,W1_PPETHM,W1_PPGENDER,W1_Q4,W1_ATTRACTION,...,W1_Q10,W1_PPREG9,W1_PPWORK,W1_Q23,W1_Q25,W1_Q26,W1_Q27,W1_Q19,W1_MARRIED,W1_Q34
0,41,41,3,1,1,1,5,2,1,1,...,1,3,1,3,1,,1,1,1,1
1,11,71,1,1,0,1,1,2,1,1,...,1,9,1,3,1,1,1,1,1,1
2,21,41,1,1,0,1,1,1,2,1,...,1,9,1,1,2,2,2,1,1,1
3,21,51,1,4,1,1,1,1,2,1,...,1,2,1,3,2,,2,1,1,1
4,41,31,1,1,0,1,1,1,2,1,...,1,8,1,1,2,2,2,,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3505,21,21,1,1,0,1,1,2,1,2,...,1,8,1,3,2,,2,1,1,1
3506,41,41,1,2,1,1,1,2,2,3,...,1,5,2,3,2,,2,2,0,1
3507,11,21,1,1,0,1,1,2,1,3,...,1,3,1,1,2,,1,2,0,1
3508,21,31,2,2,0,1,4,2,2,4,...,1,5,1,1,2,2,2,1,0,2


### Create age match column

In [168]:
df_w1_only['Same_Age_Bracket'] = (df_w1_only['W1_AGE_WHEN_MET'] == df_w1_only['W1_Q9']).astype(int)

### Create 'match' column
all of the data in this dataset is from people in relationships so they can all be considered as "matched"

In [169]:
df_w1_only['match'] = 1

### Update the Partner's Race column to include hispanic/latino

If they put they were latino in Q6A and put "Other" for Q6B then the "other" is updated to be "6" (Latino)

In [170]:
df_w1_only['W1_Q6A'] = pd.to_numeric(df_w1_only['W1_Q6A'], errors='coerce').astype('Int64')
df_w1_only['W1_Q6B'] = pd.to_numeric(df_w1_only['W1_Q6B'], errors='coerce').astype('Int64')

df_w1_only.loc[(df['W1_Q6A'] != 1) & 
       (df_w1_only['W1_Q6B'] == 5), 'W1_Q6B'] = 6

df_w1_only[['W1_Q6A','W1_Q6B']]

,W1_Q6A,W1_Q6B
0,1,1
1,1,1
2,1,1
3,1,4
4,1,1
...,...,...
3505,1,1
3506,1,2
3507,1,1
3508,1,2


### UPDATE THE SUBJECTS RACE to include hispanic/latino

In [171]:
df_w1_only['W1_PPETHM'] = pd.to_numeric(df_w1_only['W1_PPETHM'], errors='coerce').astype('Int64')
df_w1_only['W1_SUBJECT_RACE'] = pd.to_numeric(df_w1_only['W1_SUBJECT_RACE'], errors='coerce').astype('Int64')

df_w1_only.loc[(df_w1_only['W1_PPETHM'] == 4) & (df_w1_only['W1_SUBJECT_RACE'] == 5), 'W1_SUBJECT_RACE'] = 6

### Converting race columns from numeric to labels

In [172]:
race_mapping = {
    1: 'WHITE',
    2: 'BLACK',
    3: 'NATIVE AMERICAN',
    4: 'ASIAN',
    5: 'OTHER',
    6: 'LATINO'
}

df_w1_only['W1_SUBJECT_RACE'] = df_w1_only['W1_SUBJECT_RACE'].map(race_mapping)
df_w1_only['W1_Q6B'] = df_w1_only['W1_Q6B'].map(race_mapping)

### Shift Gender from 1/2 to 0/1

Changing these so they match the scale that the columbia dataset uses
Droping values of -1 (refuse to answer) and 3 (??)

In [173]:
df_w1_only['W1_Q4'] = pd.to_numeric(df_w1_only['W1_Q4'], errors='coerce').astype('Int64')
df_w1_only["W1_PPGENDER"] = pd.to_numeric(df["W1_PPGENDER"], errors='coerce').astype('Int64')

df_w1_only['W1_Q4'] = df_w1_only['W1_Q4'].replace({1: 0, 2: 1})
df_w1_only["W1_PPGENDER"] = df_w1_only["W1_PPGENDER"].replace({1: 0, 2: 1})

index_to_drop = df_w1_only[df_w1_only['W1_Q4'].isin([-1, 3])].index
df_w1_only = df_w1_only.drop(index_to_drop)
df_w1_only = df_w1_only.dropna(subset=["W1_PPGENDER"])
df_w1_only = df_w1_only.dropna(subset=["W1_Q4"])

Rename Columns to be more readable

In [174]:
df_w1_only = df_w1_only.rename(columns=target_columns_dict)

In [175]:
df_w1_only

,Subject age,Partner age,Subject race,Partner race,Interracial relationship,Hispanic origin of partner,Subject Hispanic origin,Subject gender,Partner gender,Attraction,...,Subject's employment status,Earn more money?,Same high school?,Same college?,Grew up same city?,Living Together,Married at baseline,Quality of Relationship?,Same_Age_Bracket,match
0,41,41,NATIVE AMERICAN,WHITE,1,1,5,1,0,1,...,1,3,1,,1,1,1,1,1,1
1,11,71,WHITE,WHITE,0,1,1,1,0,1,...,1,3,1,1,1,1,1,1,0,1
2,21,41,WHITE,WHITE,0,1,1,0,1,1,...,1,1,2,2,2,1,1,1,0,1
3,21,51,WHITE,ASIAN,1,1,1,0,1,1,...,1,3,2,,2,1,1,1,0,1
4,41,31,WHITE,WHITE,0,1,1,0,1,1,...,1,1,2,2,2,,0,,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3505,21,21,WHITE,WHITE,0,1,1,1,0,2,...,1,3,2,,2,1,1,1,1,1
3506,41,41,WHITE,BLACK,1,1,1,1,1,3,...,2,3,2,,2,2,0,1,1,1
3507,11,21,WHITE,WHITE,0,1,1,1,0,3,...,1,1,2,,1,2,0,1,0,1
3508,21,31,BLACK,BLACK,0,1,4,1,1,4,...,1,1,2,2,2,1,0,2,0,1


In [ ]:
df_w1_only.to_csv(output_path, index=False)

### NOTES

**Age** is masked in this survey ('W1_PPAGE' is '1' for all rows). We must use 'W1_PPAGECAT', which bins the age's

1: 18–24, \
2: 25–34, \
3: 35–44, \
4: 45–54, \
5: 55–64, \
6: 65–74, \
7: 75+

**W1_AGE_WHEN_MET** and **Parner's** Age is also binned. \
11 = 18, 19, or 20 years old, \
21 = 21–30 years old, \
31 = 31-40 years old, \
41 = 41-50 years old, \
51 = 51–60 years old, \
61 = 61-70 years old, 
